In [5]:
from pathlib import Path
import pandas as pd

In [6]:
raw_data = Path('../data/raw')
recs = pd.read_csv(raw_data/'recommendations.csv', usecols=["user_id", "app_id", "date", "is_recommended", "hours"], 
                   parse_dates=["date"])
games = pd.read_csv(raw_data / "games.csv")

print(recs.shape, games.shape)
print(recs.dtypes)
print(recs.isna().sum())
print(recs.head())

(41154794, 5) (50872, 13)
app_id                     int64
date              datetime64[us]
is_recommended              bool
hours                    float64
user_id                    int64
dtype: object
app_id            0
date              0
is_recommended    0
hours             0
user_id           0
dtype: int64
    app_id       date  is_recommended  hours  user_id
0   975370 2022-12-12            True   36.3    51580
1   304390 2017-02-17           False   11.5     2586
2  1085660 2019-11-17            True  336.5   253880
3   703080 2022-09-23            True   27.4   259432
4   526870 2021-01-10            True    7.9    23869


In [7]:
audit = {
"rows": len(recs),
"missing_hours": int(recs["hours"].isna().sum()),
"duplicate_pairs": int(recs.duplicated(["user_id", "app_id"]).sum()),
"min_hours": float(recs["hours"].min()),
"max_hours": float(recs["hours"].max()),
}
print(audit)

{'rows': 41154794, 'missing_hours': 0, 'duplicate_pairs': 21, 'min_hours': 0.0, 'max_hours': 1000.0}


In [8]:
out = Path('../data/processed/chunks')
out.mkdir(parents=True, exist_ok=True)

cols = ["user_id", "app_id", "date", "is_recommended", "hours"]
for idx, chunk in enumerate(pd.read_csv(
"../data/raw/recommendations.csv", usecols=cols,
parse_dates=["date"], chunksize=500_000)):
    chunk = chunk.dropna(subset=["user_id", "app_id", "date"])
    chunk = chunk[chunk["hours"].ge(0) & chunk["hours"].notna()]
    chunk.to_parquet(out / f"part-{idx:04d}.parquet", index=False)

In [9]:
import sys
from pathlib import Path

ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
print("cwd:", Path.cwd(), "| project root:", ROOT)

import pandas as pd
from src import preprocessing as pp
from src.utils import RAW_DIR, RESULTS_DIR, df_memory_mb, save_json
pd.set_option("display.max_columns", 50)

cwd: /Users/alllukaaa/Desktop/AI_programming_final_project/notebooks | project root: /Users/alllukaaa/Desktop/AI_programming_final_project


In [10]:
pp.check_raw_files()

,dataset,file,exists,size_mb
0,recommendations,recommendations.csv,True,1929.5
1,games,games.csv,True,4.6
2,users,users.csv,True,184.1
3,metadata,games_metadata.json,True,16.7


In [11]:
reviews = pp.load_recommendations_sample(n_rows=100_000)
print("shape:", reviews.shape, "| memory: %.1f MB" % df_memory_mb(reviews))
print(reviews.dtypes)
reviews.head()

shape: (100000, 8) | memory: 5.4 MB
app_id                     int64
helpful                    int64
funny                      int64
date              datetime64[us]
is_recommended              bool
hours                    float64
user_id                    int64
review_id                  int64
dtype: object


,app_id,helpful,funny,date,is_recommended,hours,user_id,review_id
0,975370,0,0,2022-12-12,True,36.3,51580,0
1,304390,4,0,2017-02-17,False,11.5,2586,1
2,1085660,2,0,2019-11-17,True,336.5,253880,2
3,703080,0,0,2022-09-23,True,27.4,259432,3
4,526870,0,0,2021-01-10,True,7.9,23869,4


In [12]:
pp.describe_dataframe(reviews)

,column,dtype,missing,missing_pct,n_unique,example
0,app_id,int64,0,0.0,208,975370
1,helpful,int64,0,0.0,434,0
2,funny,int64,0,0.0,250,0
3,date,datetime64[us],0,0.0,3518,2022-12-12 00:00:00
4,is_recommended,bool,0,0.0,2,True
5,hours,float64,0,0.0,9531,36.3
6,user_id,int64,0,0.0,98803,51580
7,review_id,int64,0,0.0,100000,0


In [13]:
games = pp.load_games()
print("games:", games.shape)
display(pp.describe_dataframe(games))
games.head()

games: (50872, 13)


,column,dtype,missing,missing_pct,n_unique,example
0,app_id,int64,0,0.0,50872,13500
1,title,str,0,0.0,50751,Prince of Persia: Warrior Within™
2,date_release,datetime64[us],0,0.0,4292,2008-11-21 00:00:00
3,win,bool,0,0.0,2,True
4,mac,bool,0,0.0,2,False
5,linux,bool,0,0.0,2,False
6,rating,str,0,0.0,9,Very Positive
7,positive_ratio,int64,0,0.0,100,84
8,user_reviews,int64,0,0.0,4847,2199
9,price_final,float64,0,0.0,436,9.99


,app_id,title,date_release,win,mac,linux,rating,positive_ratio,user_reviews,price_final,price_original,discount,steam_deck
0,13500,Prince of Persia: Warrior Within™,2008-11-21,True,False,False,Very Positive,84,2199,9.99,9.99,0.0,True
1,22364,BRINK: Agents of Change,2011-08-03,True,False,False,Positive,85,21,2.99,2.99,0.0,True
2,113020,Monaco: What's Yours Is Mine,2013-04-24,True,True,True,Very Positive,92,3722,14.99,14.99,0.0,True
3,226560,Escape Dead Island,2014-11-18,True,False,False,Mixed,61,873,14.99,14.99,0.0,True
4,249050,Dungeon of the ENDLESS™,2014-10-27,True,True,False,Very Positive,88,8784,11.99,11.99,0.0,True


In [14]:
users = pp.load_users()
print("users:", users.shape)
display(pp.describe_dataframe(users))
users.head()

users: (14306064, 3)


,column,dtype,missing,missing_pct,n_unique,example
0,user_id,int64,0,0.0,14306064,7360263
1,products,int64,0,0.0,7572,359
2,reviews,int64,0,0.0,681,0


,user_id,products,reviews
0,7360263,359,0
1,14020781,156,1
2,8762579,329,4
3,4820647,176,4
4,5167327,98,2


In [15]:
meta = pp.load_metadata(n_rows=1_000)
print("metadata (first 1000 lines):", meta.shape)
display(pp.describe_dataframe(meta))
meta.head()

metadata (first 1000 lines): (1000, 3)


,column,dtype,missing,missing_pct,n_unique,example
0,app_id,int64,0,0.0,1000.0,13500
1,description,str,0,0.0,755.0,Enter the dark underworld of Prince of Persia ...
2,tags,object,0,0.0,NaN,"[Action, Adventure, Parkour, Third Person, Gre..."


,app_id,description,tags
0,13500,Enter the dark underworld of Prince of Persia ...,"[Action, Adventure, Parkour, Third Person, Gre..."
1,22364,,[Action]
2,113020,Monaco: What's Yours Is Mine is a single playe...,"[Co-op, Stealth, Indie, Heist, Local Co-Op, St..."
3,226560,Escape Dead Island is a Survival-Mystery adven...,"[Zombies, Adventure, Survival, Action, Third P..."
4,249050,Dungeon of the Endless is a Rogue-Like Dungeon...,"[Roguelike, Strategy, Tower Defense, Pixel Gra..."


In [16]:
schema_check = [pp.compare_with_expected_schema(df, name) for name, df in
                [("recommendations", reviews), ("games", games), ("users", users), ("metadata", meta)]]
pd.DataFrame(schema_check)

recommendations: schema OK; missing=[] extra=[]
games: schema OK; missing=[] extra=[]
users: schema OK; missing=[] extra=[]
metadata: schema OK; missing=[] extra=[]


,dataset,missing_columns,extra_columns,ok
0,recommendations,[],[],True
1,games,[],[],True
2,users,[],[],True
3,metadata,[],[],True


In [17]:
sample_audit = pp.quality_audit(reviews, games, users)
save_json(sample_audit, RESULTS_DIR / "data_audit_sample.json")
sample_audit

{'rows': 100000,
 'missing_by_column': {'app_id': 0,
  'helpful': 0,
  'funny': 0,
  'date': 0,
  'is_recommended': 0,
  'hours': 0,
  'user_id': 0,
  'review_id': 0},
 'duplicate_full_rows': 0,
 'duplicate_pairs_user_app': 0,
 'n_users': 98803,
 'n_games': 208,
 'duplicate_review_id': 0,
 'min_hours': 0.0,
 'max_hours': 999.9,
 'median_hours': 99.2,
 'negative_hours': 0,
 'zero_hours': 2,
 'min_date': '2010-11-03',
 'max_date': '2022-12-31',
 'unparseable_dates': 0,
 'dates_before_steam_launch': 0,
 'dates_in_future': 0,
 'is_recommended_values': {'True': 84661, 'False': 15339},
 'positive_share': 0.8466,
 'reviews_with_unknown_app_id': 0,
 'unknown_app_ids': 0,
 'games_duplicate_app_id': 0,
 'reviews_with_unknown_user_id': 0,
 'users_duplicate_user_id': 0}

In [18]:
pp.cleaning_rules_table()[["problem", "rule", "reason"]]

,problem,rule,reason
0,Missing user_id / app_id / date,Drop the row,"An interaction without user, game or time cann..."
1,"Unparseable date or date outside [2003-09-12, ...",Drop the row,Wrong dates would break the temporal train/tes...
2,is_recommended is not true/false,Drop the row,The target (positive feedback) is undefined
3,hours missing or negative,Drop the row,Playtime is used as confidence weight 1 + a*lo...
4,app_id not present in games.csv,Drop the row (configurable),Game has no title/metadata and would violate t...
5,user_id not present in users.csv,Drop the row (configurable),Would violate the FK reviews.user_id -> users
6,Duplicate review_id,Keep the first occurrence,review_id is the primary key of a review
7,"Several reviews for the same (user_id, app_id)",Keep the latest by date,Steam keeps one review per user and game; the ...
8,Very large hours values (long tail),"Keep, do not clip; use log1p(hours) later","Large playtime is real behaviour, not an error"


In [19]:
before = df_memory_mb(reviews)
small = pp.clean_chunk(reviews[pp.REC_COLS].copy(), stats={})
after = df_memory_mb(pp.optimize_dtypes(small))
print(f"100k rows: {before:.2f} MB -> {after:.2f} MB ({100 * (1 - after / before):.1f}% less)")
pp.optimize_dtypes(small).dtypes

100k rows: 5.44 MB -> 2.38 MB (56.1% less)


review_id                  int32
user_id                    int32
app_id                     int32
date              datetime64[us]
is_recommended              bool
hours                    float32
dtype: object

In [20]:
audit = pp.run_preprocessing(chunksize=500_000, n_buckets=16, max_chunks=None,
                             sample_users=50_000, min_interactions=5, seed=42)

[time] convert games/users/metadata: 2.57 s
[time] load valid ids (games/users): 0.98 s
chunk    0:  500,000 ->  500,000 rows |    24.3 MB ->   11.9 MB |  0.20 s
chunk    1:  500,000 ->  500,000 rows |    24.3 MB ->   11.9 MB |  0.20 s
chunk    2:  500,000 ->  500,000 rows |    24.3 MB ->   11.9 MB |  0.25 s
chunk    3:  500,000 ->  500,000 rows |    24.3 MB ->   11.9 MB |  0.20 s
chunk    4:  500,000 ->  500,000 rows |    24.3 MB ->   11.9 MB |  0.21 s
chunk    5:  500,000 ->  500,000 rows |    24.3 MB ->   11.9 MB |  0.20 s
chunk    6:  500,000 ->  500,000 rows |    24.3 MB ->   11.9 MB |  0.20 s
chunk    7:  500,000 ->  500,000 rows |    24.3 MB ->   11.9 MB |  0.19 s
chunk    8:  500,000 ->  500,000 rows |    24.3 MB ->   11.9 MB |  0.23 s
chunk    9:  500,000 ->  500,000 rows |    24.3 MB ->   11.9 MB |  0.19 s
chunk   10:  500,000 ->  500,000 rows |    24.3 MB ->   11.9 MB |  0.20 s
chunk   11:  500,000 ->  500,000 rows |    24.3 MB ->   11.9 MB |  0.18 s
chunk   12:  500,000 -> 

In [21]:
memory_log = pd.read_csv(RESULTS_DIR / "memory_log.csv")
print(memory_log[["rows_in", "rows_out", "mem_before_mb", "mem_after_mb", "seconds"]].sum())
memory_log.head()

rows_in          41154794.00
rows_out         41154794.00
mem_before_mb        2001.77
mem_after_mb          981.13
seconds                15.21
dtype: float64


,chunk,rows_in,rows_out,mem_before_mb,mem_after_mb,seconds,process_rss_mb
0,0,500000,500000,24.32,11.92,0.202,1271.125000
1,1,500000,500000,24.32,11.92,0.203,1287.671875
2,2,500000,500000,24.32,11.92,0.249,1291.765625
3,3,500000,500000,24.32,11.92,0.195,1291.890625
4,4,500000,500000,24.32,11.92,0.206,1293.375000


In [22]:
pd.read_csv(RESULTS_DIR / "cleaning_rules.csv")

,problem,rule,reason,rows_affected
0,Missing user_id / app_id / date,Drop the row,"An interaction without user, game or time cann...",0.0
1,"Unparseable date or date outside [2003-09-12, ...",Drop the row,Wrong dates would break the temporal train/tes...,0.0
2,is_recommended is not true/false,Drop the row,The target (positive feedback) is undefined,0.0
3,hours missing or negative,Drop the row,Playtime is used as confidence weight 1 + a*lo...,0.0
4,app_id not present in games.csv,Drop the row (configurable),Game has no title/metadata and would violate t...,0.0
5,user_id not present in users.csv,Drop the row (configurable),Would violate the FK reviews.user_id -> users,0.0
6,Duplicate review_id,Keep the first occurrence,review_id is the primary key of a review,0.0
7,"Several reviews for the same (user_id, app_id)",Keep the latest by date,Steam keeps one review per user and game; the ...,21.0
8,Very large hours values (long tail),"Keep, do not clip; use log1p(hours) later","Large playtime is real behaviour, not an error",NaN


In [23]:
pd.DataFrame({
    "metric": ["raw rows", "clean rows", "kept share", "memory pandas default, MB",
               "memory optimised, MB", "reduction, %", "CSV on disk, MB", "Parquet on disk, MB"],
    "value": [audit["raw_file"]["rows"], audit["final_rows"], audit["kept_share"],
              audit["memory"]["pandas_default_mb_total"], audit["memory"]["optimized_mb_total"],
              audit["memory"]["reduction_pct"], audit["memory"]["csv_on_disk_mb"],
              audit["memory"]["parquet_on_disk_mb"]],
})

,metric,value
0,raw rows,4.115479e+07
1,clean rows,4.115477e+07
2,kept share,9.999990e-01
3,"memory pandas default, MB",2.001800e+03
4,"memory optimised, MB",9.811000e+02
5,"reduction, %",5.100000e+01
6,"CSV on disk, MB",1.929500e+03
7,"Parquet on disk, MB",4.899000e+02


In [24]:
clean = pp.load_clean_reviews()
print(clean.shape, "| %.1f MB in memory" % df_memory_mb(clean))
print(clean.dtypes)
assert not clean.duplicated(["user_id", "app_id"]).any()
assert not clean["review_id"].duplicated().any()
assert clean.notna().all().all() and (clean["hours"] >= 0).all()
clean.head()

(41154773, 6) | 981.2 MB in memory
review_id                  int32
user_id                    int32
app_id                     int32
date              datetime64[us]
is_recommended              bool
hours                    float32
dtype: object


,review_id,user_id,app_id,date,is_recommended,hours
0,19012861,1312368,8930,2010-10-15,True,40.299999
1,39171570,8447888,340,2010-10-15,True,0.000000
2,30544134,8447888,1520,2010-10-15,True,0.900000
3,18804771,8447888,4000,2010-10-15,True,75.199997
4,40837424,8447888,6040,2010-10-15,True,1.000000


In [25]:
sample = pd.read_parquet(ROOT / "data/processed/sample_reviews.parquet")
print(audit["sample"])
sample.head()

{'seed': 42, 'min_interactions': 5, 'eligible_users': 1910830, 'sampled_users': 50000, 'sample_rows': 595950, 'sample_games': 22666, 'positive_share': 0.8496, 'min_date': '2010-10-16', 'max_date': '2022-12-31'}


,review_id,user_id,app_id,date,is_recommended,hours
0,27139423,11863601,48700,2010-10-16,True,388.799988
1,20788762,11529943,8980,2010-10-16,True,202.699997
2,17114610,10331359,57300,2010-10-16,True,7.700000
3,11633596,3174301,7800,2010-10-17,True,4.000000
4,39607614,3174301,9940,2010-10-17,True,4.000000
